<a href="https://colab.research.google.com/github/TomazDrumond/Case-Valor/blob/master/Tom_NB3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TomazDrumond/Tom_Fly/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Scoring, because I have an observed outcome to learn from
(trend_direction, proxying decline), not just structure to discover (that would be clustering/unsupervised). And it's scoring rather than classification because the reviewer doesn't just need "declining: yes/no" — they need every page ranked by how urgent it is, so they can work down a queue until their capacity for the cycle runs out.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(df.shape[0], "pages | declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages | declining rate: 0.542


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The proxy over is_declining_label, derived from trend_direction == "down" in the current window flags pages currently trending down, not
pages a refresh would actually fix. A stronger version would use a forward-looking label (decline in the 30 days *after* the snapshot) rather than a same-window proxy, to avoid the label leaking information that was only knowable in hindsight.

In [2]:
print(df["trend_direction"].value_counts())
print()
print(df["is_declining_label"].value_counts(normalize=True).round(3))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@K (K=20, K=50) — because the constraint is reviewer capacity, not overall accuracy across all 30,000 pages. But precision@K on the training data isn't the score that matters — it's the "exam student" trap: a model can memorize the training pages and look great, then fail on pages it's never seen. The score that actually matters is precision@K on a client-holdout split — generalization, not memorization. This already showed up directly in my own earlier work: in-sample Precision@20 was 0.550, held-out was only 0.400 — a real example of the gap the video warns about.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
hand_rule_score = stale * visible * df["impressions_90d"]
y = df["is_declining_label"].values

print("Hand rule Precision@20:", round(precision_at_k(hand_rule_score, y, 20), 3))
print("Hand rule Precision@50:", round(precision_at_k(hand_rule_score, y, 50), 3))

Hand rule Precision@20: 0.9
Hand rule Precision@50: 0.68


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
print("One row =", "one page, at one snapshot in time")
print("Shape:", df.shape)
df[["content_age_days", "days_since_last_update", "impressions_90d",
    "avg_position", "ctr", "trend_direction", "is_declining_label"]].head()

One row = one page, at one snapshot in time
Shape: (30000, 45)


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,trend_direction,is_declining_label
0,187,20,3803,10.6,0.76,down,1
1,445,25,15320,20.3,0.05,down,1
2,141,20,12581,36.5,0.09,down,1
3,463,22,11751,6.2,0.49,stable,0
4,263,14,19140,44.0,0.13,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A **dashboard** would be enough if the reviewer just wanted to see trends and decide manually, but with thousands of pages and limited review time, eyeballing a dashboard doesn't scale to "who do I look at first."<Br>

A **rule** wins when the pattern is simple and stable: the current baseline (stale AND visible) is a reasonable first pass, cheap and transparent, but it can only combine the two thresholds it was hand-written with.<br>

**ML** is worth it here specifically because it's measured, not assumed. Moving from the fixed rule to a learned model (random forest) takes Precision@50 from 0.240 to 0.740 on the same data, per the lane guide's verified starter results — a real, measured jump.<Br>

So, a scoring model can weigh multiple weak signals (ctr, avg_position, word_count, staleness, visibility) against each other, and be re-validated on held-out data as conditions change (something a fixed if-statement can't do without a human manually re-tuning it every cycle).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.